In [5]:
import numpy as np
import pandas as pd
import cv2
import logging
import shutil
import random
import os

In [7]:
logging.basicConfig(format='\033[91m%(levelname)s: %(message)s\033[0m', level=logging.ERROR)

In [8]:
directory = "../table_tennis_data/videos/"
annotation_file = "../table_tennis_data/table_tennis.txt"
clips_dir = "../table_tennis_data/clips/"

mmaction2_ann_file_train = '../mmaction2/data/table_tennis/table_tennis_train.txt'
mmaction2_ann_file_val = '../mmaction2/data/table_tennis/table_tennis_val.txt'
mmaction2_ann_file_test = '../mmaction2/data/table_tennis/table_tennis_test.txt'

# non missing video ids
non_missing_video_ids = [413, 437, 439, 440, 444, 445, 452, 453, 457, 461, 462, 465, 466, 467, 468, 469, 472, 473, 475, 476, 477, 483, 485, 486, 487, 489, 490, 491, 492, 493, 494, 497, 501, 502, 504, 505, 506, 508, 513, 520, 521, 527, 528, 529, 530, 531, 533, 535, 536, 537, 538, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 555, 556, 560, 561, 562, 563, 564, 565, 566, 569, 570, 571, 575, 576, 577, 598, 599, 600, 601, 603, 605, 609, 610, 620, 621, 622, 623, 625, 12914, 12918, 12920, 12924, 12933, 12941, 12942, 12967, 12971, 12972, 12973, 12978, 12979, 12980, 12981, 12986, 12988, 12992, 12993, 12994, 12998]

# ssh-uploaded video ids
video_ids = [(413, '2020鄭怡靜 韓瑩.mp4'), (439, 'Tokyo Olympics Mixed Double Semifinal JPN VS TPE.mp4'), (440, 'Tokyo Olympics MIxed Double Gold Medal JPN VS CHN.mp4'), (444, '2021 Tokyo Olympics Mixed Double Bronze Medal TPE VS FRA.mp4'), (445, 'Tokyo Olympics Mixed Double Semifinal CHN VS FRA.mp4'), (453, '2021-07-27-16強賽-田志希 vs Jia Liu.mp4'), (461, '2020東奧女單四強-陳夢VS于夢雨-魏.mp4'), (462, 'Tokyo Olympics Mixed Double Quarter Final TPE VS KOR.mp4'), (465, 'Tokyo Olympics Mixed Double Quarter Final JPN VS GER.mp4'), (466, 'Tokyo Olympics Mixed Double FIrst Round TPE VS IND.mp4'), (475, 'Quarter Final CHN VS ROU.mp4'), (476, 'Quarter Final FRA VS HKG.mp4'), (477, '2020女子乒乓球世界杯半決賽，陳夢vs韩莹.mp4'), (485, 'Tokyo Olympics Fan Zhen Dong VS Lin Yun Ju.mp4'), (486, 'FIrst Round CHN VS CAN.mp4'), (487, 'Tokyo Olymipics Dimitrij Ovtcharov VS Lin Yun Ju.mp4'), (489, 'First Round FRA VS AUS.mp4'), (490, 'First Round GER VS CUB.mp4'), (491, 'First Round HKG VS HUN.mp4'), (492, 'FIrst Round JPN VS AUT.mp4'), (493, 'FIrst Round KOR VS EGY.mp4'), (494, 'First Round ROU VS SVK.mp4'), (497, '2021世錦賽鄭怡靜64強.mp4'), (521, '2022wtt8強台灣對印度.mp4'), (553, '2022匈牙利球星挑戰賽混雙冠軍賽中國對日本.mp4'), (570, '2022WTT澳門冠軍賽女單16強.mp4'), (575, '2023WTT大滿貫賽4強-台灣-中國.mp4'), (576, '2023WTT大滿貫賽8強-台灣-香港.mp4'), (577, '2023WTT大滿貫賽16強-台灣-巴西.mp4'), (600, '2023世錦賽混雙32強-中國對波蘭.mp4'), (605, '2023世錦賽混雙8強-台灣對中國.mp4'), (621, '2023WTT突尼斯挑戰賽冠亞-台灣對韓國.mp4'), (622, '2023WTT突尼斯挑戰賽4強-台灣對中國.mp4'), (625, '台灣對中國.mp4'), (12920, '2023WTT阿拉木圖挑戰賽混雙8強-台灣vs中國.mp4'), (12924, '2023WTT阿拉木圖挑戰賽混雙冠亞-台灣vs韓國.mp4'), (12941, '2023  Japan Para Open 4強.mp4'), (12981, '2323wtt蘭州公開賽-日本對韓國.mp4'), (12986, '林昀儒vs張本智和(1).mp4'), (12988, '林昀儒vs張本智和(1).mp4'), (12998, '林昀儒vsFRANZISKA.mp4')]

In [9]:
game_xlsx = "../table_tennis_data/metadata/game.xlsx"
df_game = pd.read_excel(game_xlsx, sheet_name="game")
df_game_player = pd.read_excel(game_xlsx, sheet_name="game_player")
df_game_player_item = pd.read_excel(game_xlsx, sheet_name="game_player_item")
df_game_player_double_item = pd.read_excel(game_xlsx, sheet_name="game_player_double_item")

In [ ]:
def split_video_segment(video_path, start_time, end_time, output_path):

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Cannot open video file.")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = total_frames / fps

    # Validate timestamps
    print(f"{duration}, {start_time}, {end_time}")
    if start_time < 0 or end_time > duration or start_time >= end_time:
        logging.error(f"Invalid timestamps. Video duration is {duration:.2f}s.")
        cap.release()
        return

    # Calculate frame range
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)

    # Create output directory if needed
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    # Setup video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    # Jump to the start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # Write frames in the range
    for frame_num in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)

    cap.release()
    out.release()
    print(f"Saved clip: {output_path} ({start_time:.2f}s to {end_time:.2f}s)")

In [34]:
output_clip_prefix = "clip_"
container_df = None

clip_label = []
clip_id = 0
for game_id, name in video_ids:

    # 1. Get all player_ids linked with this game_id
    player_ids = df_game_player[df_game_player["GameId"] == game_id]["Id"].tolist()

    if not player_ids:
        print(f"No players found for game_id {game_id}")
        continue

    # 2. Get game_player_item rows where gameplayerid is in player_ids
    player_item_df = df_game_player_item[df_game_player_item["GamePlayerId"].isin(player_ids)].copy()
    player_double_item_df = df_game_player_double_item[df_game_player_double_item["GamePlayerId"].isin(player_ids)].copy()

    # which df to choose from
    if not player_item_df.empty:
        container_df = player_item_df
    elif not player_double_item_df.empty:
        container_df = player_double_item_df
    else:
        print(f"No rows found for game_id {game_id}")
        continue


    # Ensure rows are sorted by id or something meaningful
    container_df = container_df.sort_values("Id").reset_index(drop=True)
    container_df["output_path"] = None

    # 3. Iterate through rows and split video based on row.iloc[22]
    for index, row in container_df.iterrows():
        print(f"Working on row {index, row.iloc[0]}")

        output_path = f"{clips_dir}{output_clip_prefix}{clip_id}.mp4"

        # First segment starts at 0
        if index == 0:
            start = 0

        end = row["Timer"]

        split_video_segment(
            directory+name,
            start_time=start,
            end_time=end,
            output_path=output_path,
        )

        # Write output path into column output_path
        container_df.loc[index, "output_path"] = output_path
        label = row["ActionId"]
        clip_filename = output_path.split("/")[-1]
        clip_label.append((clip_filename, label))

        with open(annotation_file, "a") as f:
            f.write(f"{clip_filename} {label}\n")
        
        start = end

        clip_id += 1


Working on row (0, 78132)
1416.4, 0, 20.571551
Saved clip: ../table_tennis_data/clips/clip_0.mp4 (0.00s to 20.57s)
Working on row (1, 78133)
1416.4, 20.571551, 21.573243
Saved clip: ../table_tennis_data/clips/clip_1.mp4 (20.57s to 21.57s)
Working on row (2, 78135)
1416.4, 21.573243, 22.101005
Saved clip: ../table_tennis_data/clips/clip_2.mp4 (21.57s to 22.10s)
Working on row (3, 78136)
1416.4, 22.101005, 25.258099
Saved clip: ../table_tennis_data/clips/clip_3.mp4 (22.10s to 25.26s)
Working on row (4, 78137)
1416.4, 25.258099, 26.010884
Saved clip: ../table_tennis_data/clips/clip_4.mp4 (25.26s to 26.01s)
Working on row (5, 78138)
1416.4, 26.010884, 26.717277
Saved clip: ../table_tennis_data/clips/clip_5.mp4 (26.01s to 26.72s)
Working on row (6, 78139)
1416.4, 26.717277, 27.356078
Saved clip: ../table_tennis_data/clips/clip_6.mp4 (26.72s to 27.36s)
Working on row (7, 78140)
1416.4, 27.356078, 31.48617
Saved clip: ../table_tennis_data/clips/clip_7.mp4 (27.36s to 31.49s)
Working on row (8,

ERROR: Invalid timestamps. Video duration is 641.38s.
ERROR: Invalid timestamps. Video duration is 641.38s.


Saved clip: ../table_tennis_data/clips/clip_2429.mp4 (568.73s to 569.31s)
Working on row (596, 29679)
641.3793103448276, 569.312379, 569.312379
Working on row (597, 29680)
641.3793103448276, 569.312379, 569.312379
Working on row (598, 29682)
641.3793103448276, 569.312379, 569.660794
Saved clip: ../table_tennis_data/clips/clip_2432.mp4 (569.31s to 569.66s)
Working on row (599, 29683)
641.3793103448276, 569.660794, 570.117179
Saved clip: ../table_tennis_data/clips/clip_2433.mp4 (569.66s to 570.12s)
Working on row (600, 29684)
641.3793103448276, 570.117179, 570.257192
Saved clip: ../table_tennis_data/clips/clip_2434.mp4 (570.12s to 570.26s)
Working on row (601, 29685)
641.3793103448276, 570.257192, 572.955522
Saved clip: ../table_tennis_data/clips/clip_2435.mp4 (570.26s to 572.96s)
Working on row (602, 29686)
641.3793103448276, 572.955522, 573.30296
Saved clip: ../table_tennis_data/clips/clip_2436.mp4 (572.96s to 573.30s)
Working on row (603, 29687)
641.3793103448276, 573.30296, 573.48613

ERROR: Invalid timestamps. Video duration is 1265.80s.
ERROR: Invalid timestamps. Video duration is 1265.80s.


Saved clip: ../table_tennis_data/clips/clip_6365.mp4 (283.66s to 284.11s)
Working on row (291, 83407)
1265.7966101694915, 284.109165, 284.109165
Working on row (292, 83408)
1265.7966101694915, 284.109165, 284.109165
Working on row (293, 83409)
1265.7966101694915, 284.109165, 284.620551
Saved clip: ../table_tennis_data/clips/clip_6368.mp4 (284.11s to 284.62s)
Working on row (294, 83410)
1265.7966101694915, 284.620551, 285.105098
Saved clip: ../table_tennis_data/clips/clip_6369.mp4 (284.62s to 285.11s)
Working on row (295, 83411)
1265.7966101694915, 285.105098, 286.001965
Saved clip: ../table_tennis_data/clips/clip_6370.mp4 (285.11s to 286.00s)
Working on row (296, 83412)
1265.7966101694915, 286.001965, 286.550652
Saved clip: ../table_tennis_data/clips/clip_6371.mp4 (286.00s to 286.55s)
Working on row (297, 83413)
1265.7966101694915, 286.550652, 289.542457
Saved clip: ../table_tennis_data/clips/clip_6372.mp4 (286.55s to 289.54s)
Working on row (298, 83414)
1265.7966101694915, 289.542457,

ERROR: Invalid timestamps. Video duration is 1265.80s.


Saved clip: ../table_tennis_data/clips/clip_6445.mp4 (353.54s to 363.56s)
Working on row (371, 83494)
1265.7966101694915, 363.556212, 357.599769
Working on row (372, 83495)
1265.7966101694915, 357.599769, 358.322099
Saved clip: ../table_tennis_data/clips/clip_6447.mp4 (357.60s to 358.32s)
Working on row (373, 83496)
1265.7966101694915, 358.322099, 358.920496
Saved clip: ../table_tennis_data/clips/clip_6448.mp4 (358.32s to 358.92s)
Working on row (374, 83497)
1265.7966101694915, 358.920496, 359.413986
Saved clip: ../table_tennis_data/clips/clip_6449.mp4 (358.92s to 359.41s)
Working on row (375, 83498)
1265.7966101694915, 359.413986, 363.541363
Saved clip: ../table_tennis_data/clips/clip_6450.mp4 (359.41s to 363.54s)
Working on row (376, 83499)
1265.7966101694915, 363.541363, 364.330992
Saved clip: ../table_tennis_data/clips/clip_6451.mp4 (363.54s to 364.33s)
Working on row (377, 83500)
1265.7966101694915, 364.330992, 364.990446
Saved clip: ../table_tennis_data/clips/clip_6452.mp4 (364.3

ERROR: Invalid timestamps. Video duration is 396.72s.


Saved clip: ../table_tennis_data/clips/clip_10160.mp4 (108.27s to 108.79s)
Working on row (109, 33372)
396.7241379310345, 108.78889, 108.570789
Working on row (110, 33373)
396.7241379310345, 108.570789, 112.171409
Saved clip: ../table_tennis_data/clips/clip_10162.mp4 (108.57s to 112.17s)
Working on row (111, 33374)
396.7241379310345, 112.171409, 112.687794
Saved clip: ../table_tennis_data/clips/clip_10163.mp4 (112.17s to 112.69s)
Working on row (112, 33375)
396.7241379310345, 112.687794, 113.003475
Saved clip: ../table_tennis_data/clips/clip_10164.mp4 (112.69s to 113.00s)
Working on row (113, 33376)
396.7241379310345, 113.003475, 115.670156
Saved clip: ../table_tennis_data/clips/clip_10165.mp4 (113.00s to 115.67s)
Working on row (114, 33377)
396.7241379310345, 115.670156, 116.18807
Saved clip: ../table_tennis_data/clips/clip_10166.mp4 (115.67s to 116.19s)
Working on row (115, 33378)
396.7241379310345, 116.18807, 119.305633
Saved clip: ../table_tennis_data/clips/clip_10167.mp4 (116.19s 

In [12]:
# distribute to train, val and test

train_dir = "../mmaction2/data/table_tennis/train"
val_dir = "../mmaction2/data/table_tennis/val"
test_dir = "../mmaction2/data/table_tennis/test"

# Ensure dirs exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

with open(annotation_file, "r") as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

random.shuffle(lines)
total = len(lines)
train_count = int(total * 0.8)
val_count = int(total * 0.1)
test_count = total - train_count - val_count
print(f"Number of Train clips: {train_count}, Val clips: {val_count}, Test clips: {test_count}")

train_split = lines[:train_count]
val_split = lines[train_count:train_count + val_count]
test_split = lines[train_count + val_count:]

def copy_and_write(split_lines, target_dir, annotation_path):
    """Copy video files and write annotation entries."""
    with open(annotation_path, "w") as ann_f:
        print(f"{len(split_lines), {target_dir}, {annotation_path}}")
        for entry in split_lines:
            filename, label = entry.split()

            src = os.path.join(clips_dir, filename)
            dst = os.path.join(target_dir, filename)

            # Copy the clip
            if os.path.exists(src):
                shutil.copy(src, dst)
                # Write annotation entry
                ann_f.write(f"{filename} {label}\n")
            else:
                print(f"Warning: missing clip → {src}")

copy_and_write(train_split, train_dir, mmaction2_ann_file_train)
copy_and_write(val_split, val_dir, mmaction2_ann_file_val)
copy_and_write(test_split, test_dir, mmaction2_ann_file_test)

print(
    f"Done!\n"
    f"Train: {len(train_split)} → {mmaction2_ann_file_train}\n"
    f"Val:   {len(val_split)}   → {mmaction2_ann_file_val}\n"
    f"Test:  {len(test_split)}  → {mmaction2_ann_file_test}"
)


Number of Train clips: 13511, Val clips: 1688, Test clips: 1690
(13511, {'../mmaction2/data/table_tennis/train'}, {'../mmaction2/data/table_tennis/table_tennis_train.txt'})
(1688, {'../mmaction2/data/table_tennis/val'}, {'../mmaction2/data/table_tennis/table_tennis_val.txt'})
(1690, {'../mmaction2/data/table_tennis/test'}, {'../mmaction2/data/table_tennis/table_tennis_test.txt'})
Done!
Train: 13511 → ../mmaction2/data/table_tennis/table_tennis_train.txt
Val:   1688   → ../mmaction2/data/table_tennis/table_tennis_val.txt
Test:  1690  → ../mmaction2/data/table_tennis/table_tennis_test.txt
